Generating all possible syllables in Mildredese.

First we enumerate consonants and their possible clusters.

In [14]:
con = ['m', 'n', 'nj', 'ng', 'p', 't', 'k', 'b', 'd', 'g', 'f', 'th', 's', 'kh', 'v', 'dh', 'z', 'gh', 'rh', 'r', 'l', 'h']

In [3]:
nas = 'nasal'
plo = 'plosive'
fri = 'fricative'
tap = 'tap'
manner = {'m': nas, 'n': nas, 'nj': nas, 'ng': nas,
              'p': plo, 't': plo, 'k': plo,
              'b': plo, 'd': plo, 'g': plo,
              'f': fri, 'th': fri, 's': fri, 'kh': fri,
              'v': fri, 'dh': fri, 'z': fri, 'gh': fri,
              'rh': tap, 'r': tap,
              'l': fri, 'h': fri}

In [4]:
bil = 'bilabial'
den = 'dental'
alv = 'alveolar'
pal = 'palatal'
vel = 'velar'
lat = 'lateral'
glo = 'glottal'
place = {'m': bil, 'n': den, 'nj': pal, 'ng': vel,
              'p': bil, 't': den, 'k': vel,
              'b': bil, 'd': den, 'g': vel,
              'f': bil, 'th': den, 's': alv, 'kh': vel,
              'v': bil, 'dh': den, 'z': alv, 'gh': vel,
              'rh': alv, 'r': alv,
              'l': lat, 'h': glo}

In [5]:
voicing = {'m': 1, 'n': 1, 'nj': 1, 'ng': 1,
              'p': 0, 't': 0, 'k': 0,
              'b': 1, 'd': 1, 'g': 1, # this row technically unvoiced, but for assimilation purposes treated as voiced
              'f': 0, 'th': 0, 's': 0, 'kh': 0,
              'v': 1, 'dh': 1, 'z': 1, 'gh': 1,
              'rh': 0, 'r': 1,
              'l': 0, 'h': 0}

Consonant clusters in Mildredese are plentiful. Their incidence is best described by enumerating every possible combination of two consonants, then subtracting out "incorrect" clusters based on their breaking some phonotactic rule. 

In [21]:
consonant_clusters = {c: [d for d in con] for c in con}

In [7]:
def if_then(cond, req):
    return req if cond else True

In [8]:
def apply_rule(rule, clust):
    out = {}
    for c in clust:
        out[c] = []
        for d in clust[c]:
            if rule(c,d):
                out[c].append(d)
    return out

In [12]:
consonant_rules = [
    lambda x, y: not ((place[x]==place[y]) and (manner[x]==manner[y])), # no adjacent consonants sharing PoA and MoA
    lambda x, y: not ((manner[x]==plo and manner[y]==nas) and (place[x]==place[y])), # no PLOSIVE + NASAL of same PoA
    lambda x, y: if_then((manner[x]==nas and (manner[y] in [plo, fri, tap])), (place[x]==place[y])), # NASAL assimilates to following PLOSIVE/FRICATIVE/TAP PoA
    lambda x, y: if_then(manner[x]==plo and manner[y] in [plo, fri, tap], voicing[x]==voicing[y]), # PLOSIVE aspiration assimilates to following PLOSIVE/FRICATIVE/TAP aspiration/voicing
    lambda x, y: not ((manner[x] in [nas, plo, fri]) and (manner[y]==nas and (place[y] in [pal, vel]))), # no NASAL/PLOSIVE/FRICATIVE + PALATAL/VELAR NASAL
    lambda x, y: manner[x]!=tap, # no TAP + CONSONANT
    lambda x, y: if_then(manner[x]==fri and (manner[y] in [plo, fri, tap]), voicing[x]==voicing[y]), # FRICATIVE voicing matches following PLOSIVE/FRICATIVE/TAP
    lambda x, y: not ((manner[x]==fri and place[x]==glo) or (manner[y]==fri and place[y]==glo))# no clusters including GLOTTAL FRICATIVE
]

In [10]:
def filter_clusters(clust, filters):
    filtered = clust
    for rule in filters:
        filtered = apply_rule(rule, filtered)
    return filtered

In [157]:
consonant_clusters = filter_clusters(consonant_clusters, consonant_rules)

Next, we enumerate vowels and their possible diphthongs/triphthongs. For these purposes, we categorize semivowels with the vowels.

In [ ]:
vow = ['i', 'y', 'e', 'ø', 'u', 'o', 'a']
sem = ['j', 'w']

In [18]:
hig = 'high'
mid = 'mid'
low = 'low'
height = {
    'i': hig, 'y': hig, 'e': mid, 'ø': mid, 'u': hig, 'o': mid, 'a': low, 'j': hig, 'w': hig
}

In [19]:
fro = 'front'
cen = 'central'
bac = 'back'
frontness = {
    'i': fro, 'y': fro, 'e': fro, 'ø': fro, 'u': bac, 'o': bac, 'a': cen, 'j': fro, 'w': bac
}

In [20]:
rou = 'rounded'
unr = 'unrounded'
roundedness = {
    'i': unr, 'y': rou, 'e': unr, 'ø': rou, 'u': rou, 'o': rou, 'a': unr, 'j': unr, 'w': rou
}

In [290]:
trailing_semivowel_map = {'w': 'u', 'j': 'i'} # strictly for aesthetic purposes, semivowels are represented as vowels at ends of syllables

Diphthongs and triphthongs are more limited in Mildredese and are better described positively, rather than negatively.

In [27]:
vowel_clusters = {v: [] for v in vow+sem}
vowel_clusters

{'i': [],
 'y': [],
 'e': [],
 'ø': [],
 'u': [],
 'o': [],
 'a': [],
 'j': [],
 'w': []}

In [28]:
for w in vowel_clusters:
    if w in vow:
        for s in sem:
            if not (height[w]==height[s] and frontness[w]==frontness[s]):
                vowel_clusters[w].append(s)
    elif s in sem:
        for v in vow:
            if not (height[w]==height[v] and frontness[w]==frontness[v]):
                vowel_clusters[w].append(v)
vowel_clusters

{'i': ['w'],
 'y': ['w'],
 'e': ['j', 'w'],
 'ø': ['j', 'w'],
 'u': ['j'],
 'o': ['j', 'w'],
 'a': ['j', 'w'],
 'j': ['e', 'ø', 'u', 'o', 'a'],
 'w': ['i', 'y', 'e', 'ø', 'o', 'a']}

In [51]:
import random

def bernoulli_trial(p):
    return random.random() < p

In [265]:
def random_syllable(con, vow, sem, con_clust, vow_clust):
    word = ""
    
    # syllable structure is (C)(C)(S)V(S)
    v = random.choice(vow)
    word += v

    has_trailing_semivowel = bernoulli_trial(0.1)
    if has_trailing_semivowel:
        word += trailing_semivowel_map[random.choice(vow_clust[v])]
    has_leading_semivowel = bernoulli_trial(0.3)
    if has_leading_semivowel:
        word = random.choice(vow_clust[v]) + word

    has_consonant = bernoulli_trial(0.9)
    if has_consonant:
        c = random.choice(con)
        has_consonant_cluster = False if not con_clust[c] else bernoulli_trial(0.5)
        if has_consonant_cluster:
            c += random.choice(con_clust[c])
        word = c + word
    
    print(word)

In [266]:
random_syllable(con, vow, sem, consonant_clusters, vowel_clusters)

fki


In [319]:
def enumerate_syllables(con, vow, sem, con_clust, vow_clust):
    syllables = []
    for v in vow:
        for s1 in vow_clust[v]+['']:
            for s2 in vow_clust[v]+['']:
                # print(s1, v, s2)
                for c1 in con+['']:
                    for c2 in ([''] if ((c1=='') or (not con_clust[c1])) else con_clust[c1]+['']):
                        syllables.append((c1, c2, s1, v, trailing_semivowel_map[s2] if s2 in sem else s2))
                # syllables.append((c1, c2, s1, v, s2))
    return syllables       

In [320]:
syllables = enumerate_syllables(con, vow, sem, consonant_clusters, vowel_clusters)

In [322]:
len(syllables)==len(set(syllables))

True

In [333]:
def write_syllables_to_file(syllables, filename):
    lst = []
    for syl in syllables:
        lst.append(''.join(syl)+'\n')
    with open(filename, 'w', encoding="utf-8") as file:
        file.writelines(sorted(lst))

In [334]:
write_syllables_to_file(syllables, "syllables.txt")